# Entrainement Bridge MLP: CLIP image -> CLAP text

Ce notebook entraine un projection head (MLP) pour predire l embedding textuel CLAP a partir des features visuelles CLIP sur MS COCO captions.

In [ ]:
# INSTALL DEPS WITH "uv sync" in root

In [ ]:
import io
import platform
import random
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPModel, CLIPProcessor
import laion_clap


@dataclass
class TrainConfig:
    clip_model_id: str = "openai/clip-vit-large-patch14"
    checkpoint_url: str = "https://huggingface.co/lukewys/laion_clap/resolve/main/music_audioset_epoch_15_esc_90.14.pt"
    checkpoint_path: Path = Path("checkpoints/music_audioset_epoch_15_esc_90.14.pt")

    split: str = "train"
    max_train_samples: int = 20000
    batch_size: int = 64
    num_workers: int = 4
    image_cache_dir: Path = Path("data/coco_image_cache")

    epochs: int = 5
    lr: float = 2e-4
    weight_decay: float = 1e-2
    temperature: float = 0.07
    grad_clip_norm: float = 1.0
    log_every: int = 50

    seed: int = 42

    save_path: Path = Path("artifacts/bridge_mlp_clip_to_clap.pt")

cfg = TrainConfig()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def resolve_num_workers(requested: int) -> int:
    # macOS + notebook + reseau instable avec workers multiples
    if platform.system().lower() == "darwin":
        return 0
    return max(0, requested)

def ensure_checkpoint(path: Path, url: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading CLAP checkpoint to {path}...")
        urllib.request.urlretrieve(url, path)
    else:
        print(f"Using existing CLAP checkpoint: {path}")
    return path

def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / (x.norm(dim=-1, keepdim=True) + eps)


def load_mscoco_captions(split: str):
    """Essaie plusieurs points d'entree HF pour MS COCO captions."""
    candidates: List[Tuple[str, str | None]] = [
        ("HuggingFaceM4/COCO", None),
        ("phiyodr/coco2017", None),
    ]

    last_err = None
    for repo_id, config_name in candidates:
        try:
            if config_name is None:
                ds = load_dataset(repo_id, split=split)
            else:
                ds = load_dataset(repo_id, config_name, split=split)
            print(f"Loaded dataset from {repo_id} ({split})")
            return ds
        except Exception as exc:
            last_err = exc

    raise RuntimeError(
        "Unable to load MS COCO captions from Hugging Face with known dataset ids. "
        f"Last error: {last_err}"
    )

def _pick_image_key(columns: Sequence[str]) -> str:
    for key in ["image", "img", "file_name"]:
        if key in columns:
            return key
    raise KeyError(f"No image column found. Available columns: {columns}")

def _pick_caption(sample: Dict) -> str:
    if "caption" in sample and isinstance(sample["caption"], str):
        return sample["caption"]
    if "captions" in sample:
        caps = sample["captions"]
        if isinstance(caps, list) and len(caps) > 0:
            c0 = caps[0]
            if isinstance(c0, str):
                return random.choice(caps)
            if isinstance(c0, dict):
                for cand_key in ["text", "caption", "raw"]:
                    if cand_key in c0:
                        return random.choice([x[cand_key] for x in caps if cand_key in x])
    if "sentences" in sample and isinstance(sample["sentences"], list) and len(sample["sentences"]) > 0:
        sent0 = sample["sentences"][0]
        if isinstance(sent0, dict):
            for cand_key in ["raw", "caption", "text"]:
                if cand_key in sent0:
                    pool = [x[cand_key] for x in sample["sentences"] if cand_key in x]
                    if pool:
                        return random.choice(pool)

    raise KeyError(f"No caption-like field found in sample keys: {list(sample.keys())}")

class CocoCaptionPairs(Dataset):
    def __init__(self, hf_dataset, cache_dir: Path):
        self.ds = hf_dataset
        self.image_key = _pick_image_key(self.ds.column_names)
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def __len__(self):
        return len(self.ds)

    def _download_to_cache(self, sample: Dict) -> Path:
        image_id = str(sample.get("image_id", "unknown"))
        file_name = sample.get("file_name", f"{image_id}.jpg")
        target = self.cache_dir / Path(file_name).name
        if target.exists():
            return target

        url = sample.get("coco_url") or sample.get("flickr_url")
        if not url:
            raise KeyError("No downloadable URL found in sample (missing coco_url/flickr_url).")

        urllib.request.urlretrieve(url, target)
        return target

    def _resolve_image(self, sample: Dict) -> Image.Image:
        image = sample[self.image_key]

        if isinstance(image, Image.Image):
            return image.convert("RGB")

        if isinstance(image, str):
            path = Path(image)
            if path.exists():
                return Image.open(path).convert("RGB")
            # image key may be file_name without local file; fallback to URL download
            cached = self._download_to_cache(sample)
            return Image.open(cached).convert("RGB")

        try:
            arr = np.array(image)
            if arr.ndim >= 2:
                return Image.fromarray(arr).convert("RGB")
        except Exception:
            pass

        # Last-resort fallback for metadata-only schemas
        cached = self._download_to_cache(sample)
        return Image.open(cached).convert("RGB")

    def __getitem__(self, idx: int):
        sample = self.ds[idx]
        image = self._resolve_image(sample)
        caption = _pick_caption(sample)
        return image, caption

def collate_pairs(batch):
    images = [x[0] for x in batch]
    captions = [x[1] for x in batch]
    return images, captions


In [ ]:
class BridgeMLP(nn.Module):
    def __init__(self, in_dim: int = 1024, hidden_dim: int = 1024, out_dim: int = 512, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def info_nce_loss(img_proj: torch.Tensor, txt_tgt: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    img_proj = l2_normalize(img_proj)
    txt_tgt = l2_normalize(txt_tgt)
    logits = (img_proj @ txt_tgt.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


In [ ]:
set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
effective_workers = resolve_num_workers(cfg.num_workers)
print("Device:", device)
print("DataLoader workers:", effective_workers)

clip_processor = CLIPProcessor.from_pretrained(cfg.clip_model_id)
clip_model = CLIPModel.from_pretrained(cfg.clip_model_id).to(device)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

ckpt_path = ensure_checkpoint(cfg.checkpoint_path, cfg.checkpoint_url)
clap_model = laion_clap.CLAP_Module(enable_fusion=False, amodel="HTSAT-base", device=str(device))
clap_model.load_ckpt(str(ckpt_path))
clap_model.eval()

hf_ds = load_mscoco_captions(cfg.split)
if cfg.max_train_samples is not None and cfg.max_train_samples < len(hf_ds):
    hf_ds = hf_ds.shuffle(seed=cfg.seed).select(range(cfg.max_train_samples))

train_ds = CocoCaptionPairs(hf_ds, cache_dir=cfg.image_cache_dir)
train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=effective_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_pairs,
)

bridge = BridgeMLP(in_dim=1024, hidden_dim=1024, out_dim=512, dropout=0.2).to(device)
optimizer = torch.optim.AdamW(bridge.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)


In [ ]:
bridge.train()
global_step = 0

for epoch in range(1, cfg.epochs + 1):
    running_loss = 0.0

    for images, captions in train_loader:
        with torch.no_grad():
            clip_inputs = clip_processor(images=images, return_tensors="pt")
            pixel_values = clip_inputs["pixel_values"].to(device, non_blocking=True)
            vision_out = clip_model.vision_model(pixel_values=pixel_values)
            clip_img_features = vision_out.pooler_output.float()

            clap_txt = clap_model.get_text_embedding(captions, use_tensor=True)
            if not torch.is_tensor(clap_txt):
                clap_txt = torch.tensor(clap_txt)
            clap_txt = clap_txt.to(device, non_blocking=True).float()

        pred_txt = bridge(clip_img_features)
        loss = info_nce_loss(pred_txt, clap_txt, temperature=cfg.temperature)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(bridge.parameters(), cfg.grad_clip_norm)
        optimizer.step()

        running_loss += loss.item()
        global_step += 1

        if global_step % cfg.log_every == 0:
            avg_loss = running_loss / cfg.log_every
            print(f"Epoch {epoch}/{cfg.epochs} | Step {global_step} | Loss: {avg_loss:.4f}")
            running_loss = 0.0

cfg.save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "bridge_state_dict": bridge.state_dict(),
        "config": cfg.__dict__,
        "clip_model_id": cfg.clip_model_id,
        "clap_checkpoint": str(ckpt_path),
    },
    cfg.save_path,
)
print(f"Training finished. Bridge saved to: {cfg.save_path}")